# Lensless Imaging demo

Данный ноутбук:
- клонирует репозиторий и ставит зависимости
- скачивает чекпоинт лучшей модели с HF
- скачивает по ссылке на Google Drive `.zip` с датасетом формата `CustomDir`
- запускает `inference.py`, сохраняя реконструкции
- визуализирует оригинал (если есть), lensless-изображение и реконструкцию;
- если есть `lensed/`, считает метрики через `calculate_metrics.py`.

Чтобы прогнать свой датасет - поставьте свою ссылку в последней ячейке (`DATA_URL`) и запустите все ячейки.

In [ ]:
REPO_URL = "https://github.com/tiom4eg/lensless_imaging.git"
HF_REPO = "MHDCSM/lensless-modular_prepost_ft_20260609_190232"
HF_FILENAME = "model_best.pth"
MODEL = "modular_prepost"  # which model config to instantiate

## Установка
Клонируем репозиторий и ставим зависимости из `requirements.txt`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tiom4eg/lensless_imaging.git"

repo_dir = Path("lensless_imaging")
if not repo_dir.exists():
    !git clone -q {REPO_URL} {repo_dir}
%cd {repo_dir}
!pip install -q -r requirements.txt gdown

## Скачивание чекпоинта
Чекпоинт лежит в публичном HF-репозитории и тянется через `huggingface_hub`.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

ckpt_dir = Path("saved") / MODEL
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / HF_FILENAME
if not ckpt_path.exists():
    downloaded = hf_hub_download(repo_id=HF_REPO, filename=HF_FILENAME)
    shutil.copy(downloaded, ckpt_path)
print("Checkpoint:", ckpt_path)

## Данные пользователя

Укажите прямую ссылку на `.zip` в Google Drive. Архив должен распаковываться в директорию формата `CustomDir`:

```
root/
├── lensless/ImageID.png
├── masks/ImageID.npy
└── lensed/ImageID.png   # опционально (ground truth)
```

In [ ]:
import zipfile
import gdown

def fetch_custom_dir(url, out="data/demo_input"):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    zip_path = out / "data.zip"
    gdown.download(url=url, output=str(zip_path), quiet=False, fuzzy=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out)
    # locate the directory that contains a lensless subfolder
    if (out / "lensless").is_dir():
        return out
    for sub in out.rglob("lensless"):
        if sub.is_dir():
            return sub.parent
    raise RuntimeError("No 'lensless/' folder found in the archive")

## Восстановление, визуализация и метрики
Запускаем `inference.py` на распакованном датасете, показываем примеры и (если есть `lensed/`) считаем метрики.

In [ ]:
import subprocess

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append(".")

from src.utils.image_utils import roi_crop, normalize_minmax


def _show(data_dir, pred_dir, n=4):
    ids = sorted(p.stem for p in (data_dir / "lensless").glob("*.png"))[:n]
    has_gt = (data_dir / "lensed").is_dir()
    cols = 3 if has_gt else 2
    _, axes = plt.subplots(len(ids), cols, figsize=(4 * cols, 3 * len(ids)))
    if len(ids) == 1:
        axes = axes[None, :]
    for r, image_id in enumerate(ids):
        lensless = cv2.cvtColor(cv2.imread(str(data_dir / "lensless" / f"{image_id}.png")), cv2.COLOR_BGR2RGB)
        recon = cv2.cvtColor(cv2.imread(str(pred_dir / f"{image_id}.png")), cv2.COLOR_BGR2RGB)
        recon_roi = normalize_minmax(roi_crop(torch.from_numpy(recon).permute(2, 0, 1)[None].float() / 255)).clamp(0, 1)[0].permute(1, 2, 0).numpy()
        c = 0
        if has_gt:
            gt = cv2.cvtColor(cv2.imread(str(data_dir / "lensed" / f"{image_id}.png")), cv2.COLOR_BGR2RGB)
            axes[r, c].imshow(gt)
            axes[r, c].set_title("original")
            c += 1
        axes[r, c].imshow(lensless)
        axes[r, c].set_title("lensless")
        c += 1
        axes[r, c].imshow(recon_roi)
        axes[r, c].set_title("reconstruction (ROI)")
        for a in axes[r]:
            a.axis("off")
    plt.tight_layout()
    plt.show()


def demo(url, save_path="demo_out"):
    data_dir = fetch_custom_dir(url)
    print("CustomDir:", data_dir)
    subprocess.run([
        "python", "inference.py", "datasets=custom_dir", f"model={MODEL}",
        f"datasets.test.data_dir={data_dir}",
        f"inferencer.from_pretrained={ckpt_path}",
        f"inferencer.save_path={save_path}",
        "dataloader.batch_size=1",
    ], check=True)
    pred_dir = Path("data") / save_path / "test"
    _show(data_dir, pred_dir)
    if (data_dir / "lensed").is_dir():
        print("\nMetrics on ROI:")
        subprocess.run([
            "python", "calculate_metrics.py",
            "--gt_dir", str(data_dir / "lensed"),
            "--pred_dir", str(pred_dir),
            "--device", "cuda" if torch.cuda.is_available() else "cpu",
        ], check=True)
    return pred_dir

Пример запуска:

In [ ]:
DATA_URL = ""
demo(DATA_URL)